# Validation checks

Run all cells with the project `.venv` kernel. These are the 22 checks for the dataset audit and EDA. Definitions are loaded from tagged cells in the analysis notebook; analysis execution cells are not run.

The final cell collects the visible tests through pytest using a temporary Python file, which is removed when the run finishes. All maintained project code resides in notebooks. Test results are saved in the notebook output.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import tempfile

import nbformat
import pandas as pd
import pytest

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "notebooks" / "01_dataset_audit_and_eda.ipynb").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the project folder or its notebooks directory")
analysis_path = PROJECT_ROOT / "notebooks" / "01_dataset_audit_and_eda.ipynb"
analysis = nbformat.read(analysis_path, as_version=4)
definition_sources = [
    cell.source for cell in analysis.cells
    if cell.cell_type == "code" and "definitions" in cell.metadata.get("tags", [])
]
for source in definition_sources:
    exec(compile(source, str(analysis_path), "exec"), globals())
print("Loaded analysis definitions without running the audit or creating figures.")

Loaded analysis definitions without running the audit or creating figures.


## Data validation and audit

In [2]:
@pytest.fixture
def records():
    return pd.DataFrame({
        "UDI": [1, 2, 3, 4],
        "Product ID": ["M1", "L2", "H3", "M4"],
        "Type": ["M", "L", "H", "M"],
        "Air temperature [K]": [298.1, 298.2, 299.0, 300.0],
        "Process temperature [K]": [308.6, 308.7, 309.0, 310.0],
        "Rotational speed [rpm]": [1551, 1408, 1500, 1600],
        "Torque [Nm]": [42.8, 46.3, 50.0, 40.0],
        "Tool wear [min]": [0, 3, 100, 150],
        "Machine failure": [0, 1, 1, 0],
        "TWF": [0, 1, 0, 0],
        "HDF": [0, 1, 0, 0],
        "PWF": [0, 0, 0, 0],
        "OSF": [0, 0, 0, 0],
        "RNF": [0, 0, 0, 1],
    })

In [3]:
def test_load_bom_csv_preserves_source_and_values(tmp_path, records):
    path = tmp_path / "records.csv"
    records.to_csv(path, index=False, encoding="utf-8-sig")
    before = path.read_bytes()
    result = load_dataset(path)
    pd.testing.assert_frame_equal(result, records, check_dtype=False)
    assert path.read_bytes() == before

In [4]:
def test_missing_sensor_rejected(records):
    with pytest.raises(ValueError, match="Missing required columns: Torque"):
        validate_dataset(records.drop(columns="Torque [Nm]"))

In [5]:
def test_unexpected_schema_rejected(records):
    with pytest.raises(ValueError, match="Unexpected columns: extra"):
        validate_dataset(records.assign(extra=1))

In [6]:
@pytest.mark.parametrize(("column", "value", "message"), [
    ("Type", "X", "Type must be one of"),
    ("Torque [Nm]", float("nan"), "Missing values"),
    ("Torque [Nm]", float("inf"), "nonfinite"),
    ("Torque [Nm]", "bad", "must be numeric"),
    ("Machine failure", 2, "only 0 or 1"),
    ("RNF", -1, "only 0 or 1"),
    ("Tool wear [min]", -1, "invalid nonpositive or negative"),
    ("Rotational speed [rpm]", 0, "invalid nonpositive or negative"),
    ("UDI", 1.5, "whole numbers"),
    ("Product ID", " ", "nonempty strings"),
])
def test_invalid_values_rejected(records, column, value, message):
    records[column] = records[column].astype(object)
    records.loc[0, column] = value
    records[column] = records[column].infer_objects()
    with pytest.raises(ValueError, match=message):
        validate_dataset(records)

In [7]:
def test_duplicate_csv_headers_rejected(tmp_path):
    path = tmp_path / "duplicate.csv"
    path.write_text("UDI,UDI\n1,2\n", encoding="utf-8")
    with pytest.raises(ValueError, match="Duplicate column names"):
        load_dataset(path)

In [8]:
def test_empty_dataset_rejected(records):
    with pytest.raises(ValueError, match="no records"):
        validate_dataset(records.iloc[:0])

In [9]:
def test_audit_reports_overlap_and_inconsistency_without_relabeling(records):
    original = records.copy(deep=True)
    audit = build_audit(records)
    assert audit["failure_type_counts"]["TWF"] == 1
    assert audit["failure_label_combinations"]["TWF+HDF"] == 1
    assert audit["rows_with_multiple_failure_labels"] == 1
    assert audit["failure_without_type_udi"] == [3]
    assert audit["type_without_failure_udi"] == [4]
    assert audit["target_counts"] == {0: 2, 1: 2}
    pd.testing.assert_frame_equal(records, original)

In [10]:
def test_audit_flags_extremes_without_removing_records(records):
    records.loc[3, "Torque [Nm]"] = 1000.0
    audit = build_audit(records)
    assert audit["potential_outliers_iqr"]["Torque [Nm]"]["count"] == 1
    assert audit["shape"] == [4, 14]

In [11]:
def test_audit_distinguishes_duplicate_features_from_identifiers(records):
    records = pd.concat([records, records.iloc[[0]]], ignore_index=True)
    audit = build_audit(records)
    assert audit["duplicate_rows"] == 1
    assert audit["duplicate_feature_rows"] == 1
    assert audit["duplicate_identifiers"] == {"UDI": 1, "Product ID": 1}

## EDA aggregations and preservation

In [12]:
def test_failure_rates_use_each_groups_support():
    frame = pd.DataFrame({TARGET_COLUMN: [1, 0, 0, 1, 1]})
    groups = pd.Series(["L", "L", "L", "H", "H"])
    rates = failure_rates(frame, groups)
    assert rates.loc["L", "records"] == 3
    assert rates.loc["L", "failure_rate"] == pytest.approx(1 / 3)
    assert rates.loc["H", "failure_rate"] == 1

In [13]:
def test_quantile_bins_keep_tied_readings_and_all_records():
    frame = pd.DataFrame({"sensor": [1] * 6 + [2] * 3 + [3], TARGET_COLUMN: [0] * 8 + [1, 1]})
    result = sensor_bins(frame, "sensor")
    assert result["records"].sum() == 10
    assert result["failures"].sum() == 2
    assert result.loc[0, "records"] >= 6
    assert (result["minimum"].iloc[1:].to_numpy() > result["maximum"].iloc[:-1].to_numpy()).all()

In [14]:
def test_summary_preserves_data_and_reconciles_counts():
    frame = load_dataset()
    before = frame.copy(deep=True)
    summary = build_summary(frame)
    for column in NUMERIC_COLUMNS:
        assert sum(item["records"] for item in summary["sensor_quintiles"][column]) == len(frame)
        assert sum(item["failures"] for item in summary["sensor_quintiles"][column]) == int(frame[TARGET_COLUMN].sum())
    assert sum(item["failures"] for item in summary["row_blocks"]) == int(frame[TARGET_COLUMN].sum())
    for label, support in summary["failure_label_support"].items():
        assert support["positive_flags"] == support["with_primary_failure"] + support["without_primary_failure"]
        assert summary["failure_label_cooccurrence"][label][label] == support["positive_flags"]
    pd.testing.assert_frame_equal(frame, before)

In [15]:
def test_order_analysis_uses_csv_position_and_handles_partial_block():
    frame = load_dataset().iloc[:1005].copy()
    frame["UDI"] = frame["UDI"].iloc[::-1].to_numpy()
    summary = build_summary(frame)
    first, second = summary["row_blocks"]
    assert (first["first_row"], first["last_row"], first["records"]) == (1, 1000, 1000)
    assert (second["first_row"], second["last_row"], second["records"]) == (1001, 1005, 5)
    assert first["failures"] == frame[TARGET_COLUMN].iloc[:1000].sum()
    assert second["failures"] == frame[TARGET_COLUMN].iloc[1000:].sum()

## Execute all checks

The test process uses the same Python interpreter as this notebook kernel. A failure raises an error in this cell.

In [16]:
test_notebook = nbformat.read(PROJECT_ROOT / "notebooks" / "02_validation_checks.ipynb", as_version=4)
test_sources = [
    cell.source for cell in test_notebook.cells
    if cell.cell_type == "code" and "tests" in cell.metadata.get("tags", [])
]
module_source = "\n\n".join(["import pytest", *definition_sources, *test_sources])
with tempfile.TemporaryDirectory(prefix="maintenance-tests-") as directory:
    test_path = Path(directory) / "test_notebook_checks.py"
    test_path.write_text(module_source, encoding="utf-8")
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "-q", str(test_path)],
        cwd=PROJECT_ROOT, capture_output=True, text=True,
    )
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Validation failed; inspect the pytest output above."
print("All notebook validation checks passed.")

......................                                                   [100%]
22 passed in 1.69s

All notebook validation checks passed.
